In [5]:
#%pip install pyspark

In [6]:
# Importa a classe principal para criar e iniciar a sessão Spark
from pyspark.sql import SparkSession

# Importa funções do PySpark utilizadas para limpeza e transformação dos dados
from pyspark.sql.functions import col, trim, lower, regexp_replace, when

# Pandas será usado apenas na etapa final para exportar o DataFrame tratado para CSV
import pandas as pd

# Biblioteca para uso de expressões regulares
import re

# Biblioteca para remover acentos e caracteres especiais dos textos
import unicodedata

# Biblioteca usada para verificar se o arquivo final já existe e removê-lo antes de salvar
import os


# =========================
# 1. CRIAÇÃO DA SESSÃO SPARK
# =========================

# Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("WeatherForecastCleaning") \
    .getOrCreate()


# =========================
# 2. DEFINIÇÃO DOS CAMINHOS
# =========================

# Arquivo de entrada (dados brutos)
arquivo_entrada = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016.csv"

# Arquivos de saída
arquivo_saida_texto = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016_clean_texto.csv"
arquivo_saida_numero = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016_clean_numero.csv"


# =========================
# 3. EXTRAÇÃO (E DO ETL)
# =========================

# Lê o CSV como string para ter controle total na transformação
df = spark.read.option("header", True).option("inferSchema", False).csv(arquivo_entrada)


# =========================
# 4. FUNÇÃO PARA SNAKE_CASE
# =========================

# Função para padronizar nomes de colunas
def to_snake_case(text):
    if text is None:
        return None

    # Remove espaços e coloca em minúsculo
    text = str(text).strip().lower()

    # Remove acentos
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")

    # Remove caracteres especiais
    text = re.sub(r"[^\w\s]", "", text)

    # Substitui espaços por _
    text = re.sub(r"\s+", "_", text)

    return text


# =========================
# 5. TRADUÇÃO DAS COLUNAS
# =========================

# Mapeamento PT → EN
mapeamento_colunas = {
    "Data": "date",
    "Temperatura Máxima": "max_temperature",
    "Temperatura Mínima": "min_temperature",
    "Chuva (mm)": "rain_mm",
    "Código do Clima": "weather_code"
}

# Renomeia se existir
for coluna_antiga, coluna_nova in mapeamento_colunas.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)


# =========================
# 6. PADRONIZAÇÃO DAS COLUNAS
# =========================

# Aplica snake_case em todas as colunas
df = df.toDF(*[to_snake_case(c) for c in df.columns])


# =========================
# 7. PADRONIZAÇÃO DOS DADOS STRING
# =========================

# Padroniza valores string
for campo, tipo in df.dtypes:
    if tipo == "string":
        df = df.withColumn(campo, trim(lower(col(campo))))
        df = df.withColumn(campo, regexp_replace(col(campo), r"\s+", "_"))


# =========================
# 8. TRATAMENTO DE NULOS
# =========================

# Converte strings vazias para null
for campo in df.columns:
    df = df.withColumn(
        campo,
        when(trim(col(campo)) == "", None).otherwise(col(campo))
    )


# =========================
# 9. CRIAÇÃO DA DESCRIÇÃO DO CLIMA
# =========================

# Cria coluna textual baseada no código
if "weather_code" in df.columns:
    df = df.withColumn(
        "weather_description",

        when(col("weather_code") == "0", "clear_sky")
        .when(col("weather_code") == "1", "mainly_clear")
        .when(col("weather_code") == "2", "partly_cloudy")
        .when(col("weather_code") == "3", "overcast")
        .when(col("weather_code") == "45", "fog")
        .when(col("weather_code") == "48", "depositing_rime_fog")
        .when(col("weather_code") == "51", "light_drizzle")
        .when(col("weather_code") == "53", "moderate_drizzle")
        .when(col("weather_code") == "55", "dense_drizzle")
        .when(col("weather_code") == "56", "light_freezing_drizzle")
        .when(col("weather_code") == "57", "dense_freezing_drizzle")
        .when(col("weather_code") == "61", "light_rain")
        .when(col("weather_code") == "63", "moderate_rain")
        .when(col("weather_code") == "65", "heavy_rain")
        .when(col("weather_code") == "66", "light_freezing_rain")
        .when(col("weather_code") == "67", "heavy_freezing_rain")
        .when(col("weather_code") == "71", "light_snow_fall")
        .when(col("weather_code") == "73", "moderate_snow_fall")
        .when(col("weather_code") == "75", "heavy_snow_fall")
        .when(col("weather_code") == "77", "snow_grains")
        .when(col("weather_code") == "80", "light_rain_showers")
        .when(col("weather_code") == "81", "moderate_rain_showers")
        .when(col("weather_code") == "82", "violent_rain_showers")
        .when(col("weather_code") == "85", "light_snow_showers")
        .when(col("weather_code") == "86", "heavy_snow_showers")
        .when(col("weather_code") == "95", "thunderstorm")
        .when(col("weather_code") == "96", "thunderstorm_with_light_hail")
        .when(col("weather_code") == "99", "thunderstorm_with_heavy_hail")
        .otherwise("unknown")
    )


# =========================
# 10. REMOVER DUPLICADOS
# =========================

df = df.dropDuplicates()


# =========================
# 11. CRIAÇÃO DOS DATASETS
# =========================

# Dataset com descrição textual
df_texto = df

# Dataset numérico → REMOVE a descrição (como você pediu)
df_numero = df.drop("weather_description")


# =========================
# 12. VALIDAÇÃO
# =========================

df.printSchema()
df.limit(10).show(truncate=False)


# =========================
# 13. EXPORTAÇÃO
# =========================

# Converte para pandas
df_texto_pandas = df_texto.toPandas()
df_numero_pandas = df_numero.toPandas()

try:
    # Remove arquivos antigos
    if os.path.exists(arquivo_saida_texto):
        os.remove(arquivo_saida_texto)

    if os.path.exists(arquivo_saida_numero):
        os.remove(arquivo_saida_numero)

    # Exporta CSVs
    df_texto_pandas.to_csv(arquivo_saida_texto, index=False, encoding="utf-8")
    df_numero_pandas.to_csv(arquivo_saida_numero, index=False, encoding="utf-8")

    print(f"Arquivo com descrição salvo em: {arquivo_saida_texto}")
    print(f"Arquivo numérico salvo em: {arquivo_saida_numero}")

except PermissionError:
    print("Erro: não foi possível salvar os arquivos. Verifique se estão abertos.")

root
 |-- date: string (nullable = true)
 |-- max_temperature: string (nullable = true)
 |-- min_temperature: string (nullable = true)
 |-- rain_mm: string (nullable = true)
 |-- weather_code: string (nullable = true)
 |-- weather_description: string (nullable = false)

+----------+---------------+---------------+-------+------------+-------------------+
|date      |max_temperature|min_temperature|rain_mm|weather_code|weather_description|
+----------+---------------+---------------+-------+------------+-------------------+
|2016-09-05|20.9           |16.0           |1.5    |55          |dense_drizzle      |
|2016-10-02|19.2           |13.6           |2.7    |51          |light_drizzle      |
|2016-10-17|31.1           |21.4           |0.3    |51          |light_drizzle      |
|2016-05-04|21.3           |13.8           |0.0    |3           |overcast           |
|2016-08-23|17.3           |8.3            |0.0    |3           |overcast           |
|2016-10-29|18.7           |11.5         

In [ ]:
# Importa a classe principal para criar e iniciar a sessão Spark
from pyspark.sql import SparkSession

# Importa funções do PySpark utilizadas para limpeza e transformação dos dados
from pyspark.sql.functions import col, trim, lower, regexp_replace, when

# Pandas será usado apenas na etapa final para exportar o DataFrame tratado para CSV
import pandas as pd

# Biblioteca para uso de expressões regulares
import re

# Biblioteca para remover acentos e caracteres especiais dos textos
import unicodedata

# Biblioteca usada para verificar se o arquivo final já existe e removê-lo antes de salvar
import os


# =========================
# 1. CRIAÇÃO DA SESSÃO SPARK
# =========================

# Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("WeatherForecastCleaning") \
    .getOrCreate()


# =========================
# 2. DEFINIÇÃO DOS CAMINHOS
# =========================

# Arquivo de entrada (dados brutos)
arquivo_entrada = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016.csv"

# Arquivos de saída
arquivo_saida_texto = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016_clean_texto.csv"
arquivo_saida_numero = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\clima_2016_clean_numero.csv"


# =========================
# 3. EXTRAÇÃO (E DO ETL)
# =========================

# Lê o CSV como string para ter controle total na transformação
df = spark.read.option("header", True).option("inferSchema", False).csv(arquivo_entrada)


# =========================
# 4. FUNÇÃO PARA SNAKE_CASE
# =========================

# Função para padronizar nomes de colunas
def to_snake_case(text):
    if text is None:
        return None

    # Remove espaços e coloca em minúsculo
    text = str(text).strip().lower()

    # Remove acentos
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")

    # Remove caracteres especiais
    text = re.sub(r"[^\w\s]", "", text)

    # Substitui espaços por _
    text = re.sub(r"\s+", "_", text)

    return text


# =========================
# 5. TRADUÇÃO DAS COLUNAS
# =========================

# Mapeamento PT → EN
mapeamento_colunas = {
    "Data": "date",
    "Temperatura Máxima": "max_temperature",
    "Temperatura Mínima": "min_temperature",
    "Chuva (mm)": "rain_mm",
    "Código do Clima": "weather_code"
}

# Renomeia se existir
for coluna_antiga, coluna_nova in mapeamento_colunas.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)


# =========================
# 6. PADRONIZAÇÃO DAS COLUNAS
# =========================

# Aplica snake_case em todas as colunas
df = df.toDF(*[to_snake_case(c) for c in df.columns])


# =========================
# 7. PADRONIZAÇÃO DOS DADOS STRING
# =========================

# Padroniza valores string
for campo, tipo in df.dtypes:
    if tipo == "string":
        df = df.withColumn(campo, trim(lower(col(campo))))
        df = df.withColumn(campo, regexp_replace(col(campo), r"\s+", "_"))


# =========================
# 8. TRATAMENTO DE NULOS
# =========================

# Converte strings vazias para null
for campo in df.columns:
    df = df.withColumn(
        campo,
        when(trim(col(campo)) == "", None).otherwise(col(campo))
    )


# =========================
# 9. CRIAÇÃO DA DESCRIÇÃO DO CLIMA
# =========================

# Cria coluna textual baseada no código
if "weather_code" in df.columns:
    df = df.withColumn(
        "weather_description",

        when(col("weather_code") == "0", "clear_sky")
        .when(col("weather_code") == "1", "mainly_clear")
        .when(col("weather_code") == "2", "partly_cloudy")
        .when(col("weather_code") == "3", "overcast")
        .when(col("weather_code") == "45", "fog")
        .when(col("weather_code") == "48", "depositing_rime_fog")
        .when(col("weather_code") == "51", "light_drizzle")
        .when(col("weather_code") == "53", "moderate_drizzle")
        .when(col("weather_code") == "55", "dense_drizzle")
        .when(col("weather_code") == "56", "light_freezing_drizzle")
        .when(col("weather_code") == "57", "dense_freezing_drizzle")
        .when(col("weather_code") == "61", "light_rain")
        .when(col("weather_code") == "63", "moderate_rain")
        .when(col("weather_code") == "65", "heavy_rain")
        .when(col("weather_code") == "66", "light_freezing_rain")
        .when(col("weather_code") == "67", "heavy_freezing_rain")
        .when(col("weather_code") == "71", "light_snow_fall")
        .when(col("weather_code") == "73", "moderate_snow_fall")
        .when(col("weather_code") == "75", "heavy_snow_fall")
        .when(col("weather_code") == "77", "snow_grains")
        .when(col("weather_code") == "80", "light_rain_showers")
        .when(col("weather_code") == "81", "moderate_rain_showers")
        .when(col("weather_code") == "82", "violent_rain_showers")
        .when(col("weather_code") == "85", "light_snow_showers")
        .when(col("weather_code") == "86", "heavy_snow_showers")
        .when(col("weather_code") == "95", "thunderstorm")
        .when(col("weather_code") == "96", "thunderstorm_with_light_hail")
        .when(col("weather_code") == "99", "thunderstorm_with_heavy_hail")
        .otherwise("unknown")
    )


# =========================
# 10. REMOVER DUPLICADOS
# =========================

df = df.dropDuplicates()


# =========================
# 11. CRIAÇÃO DOS DATASETS
# =========================

# Dataset com descrição textual
df_texto = df

# Dataset numérico → REMOVE a descrição (como você pediu)
df_numero = df.drop("weather_description")


# =========================
# 12. VALIDAÇÃO
# =========================

df.printSchema()
df.limit(10).show(truncate=False)


# =========================
# 13. EXPORTAÇÃO
# =========================

# Converte para pandas
df_texto_pandas = df_texto.toPandas()
df_numero_pandas = df_numero.toPandas()

try:
    # Remove arquivos antigos
    if os.path.exists(arquivo_saida_texto):
        os.remove(arquivo_saida_texto)

    if os.path.exists(arquivo_saida_numero):
        os.remove(arquivo_saida_numero)

    # Exporta CSVs
    df_texto_pandas.to_csv(arquivo_saida_texto, index=False, encoding="utf-8")
    df_numero_pandas.to_csv(arquivo_saida_numero, index=False, encoding="utf-8")

    print(f"Arquivo com descrição salvo em: {arquivo_saida_texto}")
    print(f"Arquivo numérico salvo em: {arquivo_saida_numero}")

except PermissionError:
    print("Erro: não foi possível salvar os arquivos. Verifique se estão abertos.")

root
 |-- date: string (nullable = true)
 |-- max_temperature: string (nullable = true)
 |-- min_temperature: string (nullable = true)
 |-- rain_mm: string (nullable = true)
 |-- weather_code: string (nullable = true)
 |-- weather_description: string (nullable = false)

+----------+---------------+---------------+-------+------------+-------------------+
|date      |max_temperature|min_temperature|rain_mm|weather_code|weather_description|
+----------+---------------+---------------+-------+------------+-------------------+
|2016-09-05|20.9           |16.0           |1.5    |55          |dense_drizzle      |
|2016-10-02|19.2           |13.6           |2.7    |51          |light_drizzle      |
|2016-10-17|31.1           |21.4           |0.3    |51          |light_drizzle      |
|2016-05-04|21.3           |13.8           |0.0    |3           |overcast           |
|2016-08-23|17.3           |8.3            |0.0    |3           |overcast           |
|2016-10-29|18.7           |11.5         